In [7]:
# ============================================================
# V2B Incremental Saving Dataset Generator - Version 2
# Existing PV Building / S0-S4 Scenario Version
# ============================================================

import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd


# ============================================================
# 0. Global Settings
# ============================================================

T = range(24)
PEAK_HOURS = range(16, 22)

building_area = 1081.65

# EV parameters
ev_capacity_single = 67.5   # kWh per EV
ev_ch_max_single = 7        # kW per EV
ev_dis_max_single = 7       # kW per EV

ev_station_ch_limit = 110   # kW total charging limit
ev_station_dis_limit = 35   # kW total discharging limit

# Building battery parameters
bat_cap_single = 13.5       # kWh per battery
bat_ch_single = 5           # kW per battery
bat_dis_single = 5          # kW per battery

bat_init_pct = 0.2
bat_min_pct = 0.2
bat_max_pct = 0.9

# Efficiency
eta_ch = 0.95
eta_dis = 0.95

# Degradation cost
ev_deg_cost = 0
bat_deg_cost = 0

# Electricity price
cheapest_price = 2.23


# ============================================================
# 1. Load Fixed Data
# ============================================================

def load_building_load(meter_file="0512電表.csv"):
    df_meter = pd.read_csv(meter_file, header=1)

    bh = pd.to_numeric(df_meter.iloc[:, 1], errors="coerce")
    bh = bh.ffill().bfill().values[:24]

    if len(bh) < 24:
        raise ValueError("Building load data length is less than 24 hours.")

    return bh


def calculate_pv_from_weather(weather_file):
    """
    Read CWB weather file and calculate hourly PV generation.
    """

    df_weather = pd.read_csv(weather_file, header=0)
    df_weather = df_weather.drop(0).reset_index(drop=True)

    t_amb_data = pd.to_numeric(
        df_weather["氣溫(℃)"],
        errors="coerce"
    ).ffill().bfill().values[:24]

    rad_mj_data = pd.to_numeric(
        df_weather["全天空日射量(MJ/㎡)"],
        errors="coerce"
    ).fillna(0).values[:24]

    if len(t_amb_data) < 24 or len(rad_mj_data) < 24:
        raise ValueError(f"Weather data length is less than 24 hours: {weather_file}")

    # 1 MJ/m2/hour = 1/3.6 kW/m2
    rad_kw_data = rad_mj_data / 3.6

    T_ref = 25.0
    K_pv = -3.7e-3
    eta_pv = 0.22

    rh = np.zeros(24)

    for t in T:
        if rad_kw_data[t] > 0:
            Rad_W = rad_kw_data[t] * 1000

            T_pv = t_amb_data[t] + 0.0256 * Rad_W

            p_pv_t = (
                rad_kw_data[t]
                * building_area
                * (1 + K_pv * (T_pv - T_ref))
                * eta_pv
            )

            rh[t] = max(0, p_pv_t)
        else:
            rh[t] = 0

    return rh


# ============================================================
# 2. Scenario Generator
# ============================================================

def generate_price(price_gap):
    """
    price_gap range: 0 to 5

    0-8   : off-peak
    9-15  : mid-peak
    16-21 : peak
    22-23 : mid-peak

    Note:
    max_price - min_price = 2 * price_gap
    """

    price = np.array(
        [cheapest_price] * 9
        + [cheapest_price + price_gap] * 7
        + [cheapest_price + 2 * price_gap] * 6
        + [cheapest_price + price_gap] * 2
    )

    return price


def generate_pv_profile(
    rh_sunny_base,
    rh_rainy_base,
    weather_type,
    PV_install_rate
):
    """
    weather_type:
    1 = sunny
    0 = rainy
    """

    if weather_type == 1:
        rh = rh_sunny_base * PV_install_rate
    else:
        rh = rh_rainy_base * PV_install_rate

    return rh


def generate_ev_data(ev_count):
    """
    Generate EV arrival/departure time and SOC.
    """

    if ev_count > 0:
        lambda_arr = ev_count / 2.0
        lambda_dep = ev_count / 4.0

        # Arrival time
        u_arr = np.random.uniform(0, 1, ev_count)
        inter_arrival_times = -np.log(1 - u_arr) / lambda_arr

        arrival_times = 7.5 + np.cumsum(inter_arrival_times)
        arrival_times = np.round(arrival_times).astype(int)
        arrival_times = np.clip(arrival_times, 0, 23)

        # Departure time
        u_dep = np.random.uniform(0, 1, ev_count)
        inter_departure_times = -np.log(1 - u_dep) / lambda_dep

        departure_times = 16.5 + np.cumsum(inter_departure_times)
        departure_times = np.round(departure_times).astype(int)

        departure_times = np.maximum(departure_times, arrival_times + 1)
        departure_times = np.clip(departure_times, 1, 24)

        # SOC
        soc_init_pcts = np.random.uniform(0.1, 0.4, size=ev_count)
        soc_target_pcts = np.full(ev_count, 0.9)

    else:
        arrival_times = np.array([])
        departure_times = np.array([])
        soc_init_pcts = np.array([])
        soc_target_pcts = np.array([])

    return arrival_times, departure_times, soc_init_pcts, soc_target_pcts


# ============================================================
# 3. PV-only Baseline
# ============================================================

def simulate_unmanaged_ev_charging(
    ev_count,
    arrival_times,
    departure_times,
    soc_init_pcts,
    soc_target_pcts
):
    """
    Unmanaged EV charging:
    - EV starts charging immediately after arrival.
    - EV charges until target SOC.
    - No smart scheduling.
    - No EV discharging.
    """

    ev_charge_profile = np.zeros(24)

    if ev_count == 0:
        return ev_charge_profile, 0.0, 0

    ev_soc_kWh = soc_init_pcts * ev_capacity_single
    ev_target_kWh = soc_target_pcts * ev_capacity_single

    for t in T:
        station_remaining = ev_station_ch_limit

        for i in range(ev_count):
            if arrival_times[i] <= t < departure_times[i]:
                remaining_need_kWh = ev_target_kWh[i] - ev_soc_kWh[i]

                if remaining_need_kWh > 1e-6 and station_remaining > 1e-6:
                    grid_side_energy_need = remaining_need_kWh / eta_ch

                    charge_power = min(
                        ev_ch_max_single,
                        station_remaining,
                        grid_side_energy_need
                    )

                    ev_charge_profile[t] += charge_power
                    ev_soc_kWh[i] += charge_power * eta_ch
                    station_remaining -= charge_power

    ev_unmet_kWh = np.maximum(0, ev_target_kWh - ev_soc_kWh)
    ev_unmet_total = np.sum(ev_unmet_kWh)
    ev_unmet_count = int(np.sum(ev_unmet_kWh > 1e-6))

    return ev_charge_profile, ev_unmet_total, ev_unmet_count


def calculate_pv_only_unmanaged_baseline(
    rh,
    bh,
    price,
    ev_count,
    arrival_times,
    departure_times,
    soc_init_pcts,
    soc_target_pcts
):
    """
    New S0 baseline:

    Existing PV building.
    No fixed battery.
    No EV smart charging.
    No EV discharging.
    EVs charge immediately after arrival.

    PV offsets simultaneous total load:
    total_load = building_load + unmanaged_EV_charging
    """

    unmanaged_ev_charge, ev_unmet_kWh, ev_unmet_count = simulate_unmanaged_ev_charging(
        ev_count=ev_count,
        arrival_times=arrival_times,
        departure_times=departure_times,
        soc_init_pcts=soc_init_pcts,
        soc_target_pcts=soc_target_pcts
    )

    total_fixed_load = bh + unmanaged_ev_charge

    grid_profile = np.maximum(total_fixed_load - rh, 0)
    pv_used_profile = np.minimum(total_fixed_load, rh)
    pv_curt_profile = np.maximum(rh - total_fixed_load, 0)

    total_cost = np.sum(grid_profile * price)

    result = {
        "total_cost": total_cost,
        "grid_import_total_kWh": np.sum(grid_profile),
        "grid_import_peak_kW": np.max(grid_profile),
        "pv_self_consumption_kWh": np.sum(pv_used_profile),
        "pv_curtailment_kWh": np.sum(pv_curt_profile),

        "ev_charge_total": np.sum(unmanaged_ev_charge),
        "ev_discharge_total": 0.0,
        "ev_unmet_kWh": ev_unmet_kWh,
        "ev_departure_violation_count": ev_unmet_count,

        "bat_charge_total": 0.0,
        "bat_discharge_total": 0.0,
        "battery_cycles": 0.0
    }

    return result


# ============================================================
# 4. Scenario Config
# ============================================================

SCENARIOS = {
    "S1_PV_battery": {
        "enable_battery": True,
        "enable_smart_ev_charging": False,
        "enable_ev_discharge": False
    },

    "S2_PV_smart_EV_charging": {
        "enable_battery": False,
        "enable_smart_ev_charging": True,
        "enable_ev_discharge": False
    },

    "S3_PV_V2B": {
        "enable_battery": False,
        "enable_smart_ev_charging": True,
        "enable_ev_discharge": True
    },

    "S4_PV_battery_V2B": {
        "enable_battery": True,
        "enable_smart_ev_charging": True,
        "enable_ev_discharge": True
    }
}


# ============================================================
# 5. Gurobi Optimization
# ============================================================

def run_v2b_optimization(
    rh,
    bh,
    price,
    ev_count,
    bat_count,
    arrival_times,
    departure_times,
    soc_init_pcts,
    soc_target_pcts,
    scenario_config
):
    """
    Flexible optimizer for S1-S4.

    If enable_smart_ev_charging = False:
        EV charging is unmanaged and treated as fixed load.

    If enable_smart_ev_charging = True:
        EV charging is optimized.

    If enable_ev_discharge = True:
        EV can discharge to building load.

    If enable_battery = True:
        Fixed battery can charge/discharge.
    """

    enable_battery = scenario_config["enable_battery"]
    enable_smart_ev = scenario_config["enable_smart_ev_charging"]
    enable_ev_discharge = scenario_config["enable_ev_discharge"]

    # --------------------------------------------------------
    # Fixed unmanaged EV load if smart EV is disabled
    # --------------------------------------------------------

    if enable_smart_ev:
        unmanaged_ev_charge = np.zeros(24)
        unmanaged_ev_unmet_kWh = 0.0
        unmanaged_ev_unmet_count = 0
    else:
        unmanaged_ev_charge, unmanaged_ev_unmet_kWh, unmanaged_ev_unmet_count = simulate_unmanaged_ev_charging(
            ev_count=ev_count,
            arrival_times=arrival_times,
            departure_times=departure_times,
            soc_init_pcts=soc_init_pcts,
            soc_target_pcts=soc_target_pcts
        )

    # --------------------------------------------------------
    # Battery limits
    # --------------------------------------------------------

    if enable_battery and bat_count > 0:
        bat_cap_real = bat_cap_single * bat_count
        bat_ch_max = bat_ch_single * bat_count
        bat_dis_max = bat_dis_single * bat_count
    else:
        bat_cap_real = 0.0
        bat_ch_max = 0.0
        bat_dis_max = 0.0

    bat_cap_model = bat_cap_real if bat_cap_real > 0 else 1.0

    # --------------------------------------------------------
    # EV optimization range
    # --------------------------------------------------------

    if enable_smart_ev:
        I = range(ev_count)
    else:
        I = range(0)

    ev_dis_ub = ev_dis_max_single if enable_ev_discharge else 0.0

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    m = gp.Model("V2B_Incremental_Saving")
    m.Params.OutputFlag = 0

    # Grid
    grid = m.addVars(T, lb=0, name="grid")

    # PV allocation
    pv_to_load = m.addVars(T, lb=0, name="pv_to_load")
    pv_to_ev = m.addVars(T, lb=0, name="pv_to_ev")
    pv_to_bat = m.addVars(T, lb=0, name="pv_to_bat")
    pv_curt = m.addVars(T, lb=0, name="pv_curt")

    # Grid allocation
    grid_to_load = m.addVars(T, lb=0, name="grid_to_load")
    grid_to_ev = m.addVars(T, lb=0, name="grid_to_ev")
    grid_to_bat = m.addVars(T, lb=0, name="grid_to_bat")

    # EV variables
    ev_ch = m.addVars(I, T, lb=0, ub=ev_ch_max_single, name="ev_ch")
    ev_dis = m.addVars(I, T, lb=0, ub=ev_dis_ub, name="ev_dis")
    soc_ev = m.addVars(I, range(25), lb=0.0, ub=1.0, name="soc_ev")
    ev_mode = m.addVars(I, T, vtype=GRB.BINARY, name="ev_mode")

    # Battery variables
    bat_ch = m.addVars(T, lb=0, ub=bat_ch_max, name="bat_ch")
    bat_dis = m.addVars(T, lb=0, ub=bat_dis_max, name="bat_dis")
    soc_bat = m.addVars(range(25), lb=bat_min_pct, ub=bat_max_pct, name="soc_bat")
    bat_mode = m.addVars(T, vtype=GRB.BINARY, name="bat_mode")

    # --------------------------------------------------------
    # Initial SOC
    # --------------------------------------------------------

    for i in I:
        m.addConstr(
            soc_ev[i, 0] == soc_init_pcts[i],
            name=f"ev_init_{i}"
        )

    m.addConstr(
        soc_bat[0] == bat_init_pct,
        name="bat_init"
    )

    # --------------------------------------------------------
    # Main constraints
    # --------------------------------------------------------

    for t in T:

        fixed_load_t = bh[t] + unmanaged_ev_charge[t]

        # PV balance
        m.addConstr(
            pv_to_load[t]
            + pv_to_ev[t]
            + pv_to_bat[t]
            + pv_curt[t]
            == rh[t],
            name=f"pv_balance_{t}"
        )

        # Building or fixed-load balance
        m.addConstr(
            pv_to_load[t]
            + grid_to_load[t]
            + gp.quicksum(ev_dis[i, t] for i in I)
            + bat_dis[t]
            == fixed_load_t,
            name=f"load_balance_{t}"
        )

        # Smart EV charging source balance
        if enable_smart_ev:
            m.addConstr(
                gp.quicksum(ev_ch[i, t] for i in I)
                == pv_to_ev[t] + grid_to_ev[t],
                name=f"ev_charge_source_{t}"
            )
        else:
            m.addConstr(pv_to_ev[t] == 0, name=f"pv_to_ev_zero_{t}")
            m.addConstr(grid_to_ev[t] == 0, name=f"grid_to_ev_zero_{t}")

        # Battery charging source balance
        m.addConstr(
            bat_ch[t] == pv_to_bat[t] + grid_to_bat[t],
            name=f"bat_charge_source_{t}"
        )

        # Grid total
        m.addConstr(
            grid[t]
            == grid_to_load[t]
            + grid_to_ev[t]
            + grid_to_bat[t],
            name=f"grid_total_{t}"
        )

        # Station limits
        if enable_smart_ev:
            m.addConstr(
                gp.quicksum(ev_ch[i, t] for i in I)
                <= ev_station_ch_limit,
                name=f"station_ch_limit_{t}"
            )

            m.addConstr(
                gp.quicksum(ev_dis[i, t] for i in I)
                <= ev_station_dis_limit,
                name=f"station_dis_limit_{t}"
            )

        # Battery SOC update
        m.addConstr(
            soc_bat[t + 1]
            == soc_bat[t]
            + bat_ch[t] * eta_ch / bat_cap_model
            - bat_dis[t] / (eta_dis * bat_cap_model),
            name=f"bat_soc_update_{t}"
        )

        # Battery charge/discharge mode
        m.addConstr(
            bat_ch[t] <= bat_ch_max * bat_mode[t],
            name=f"bat_ch_mode_{t}"
        )

        m.addConstr(
            bat_dis[t] <= bat_dis_max * (1 - bat_mode[t]),
            name=f"bat_dis_mode_{t}"
        )

        # EV constraints
        for i in I:

            # EV SOC update
            m.addConstr(
                soc_ev[i, t + 1]
                == soc_ev[i, t]
                + ev_ch[i, t] * eta_ch / ev_capacity_single
                - ev_dis[i, t] / (eta_dis * ev_capacity_single),
                name=f"ev_soc_update_{i}_{t}"
            )

            # EV unavailable outside parking time
            if t < arrival_times[i] or t >= departure_times[i]:
                m.addConstr(
                    ev_ch[i, t] == 0,
                    name=f"ev_ch_unavailable_{i}_{t}"
                )

                m.addConstr(
                    ev_dis[i, t] == 0,
                    name=f"ev_dis_unavailable_{i}_{t}"
                )

                m.addConstr(
                    soc_ev[i, t + 1] == soc_ev[i, t],
                    name=f"ev_soc_hold_{i}_{t}"
                )

            # EV charge/discharge mode
            m.addConstr(
                ev_ch[i, t] <= ev_ch_max_single * ev_mode[i, t],
                name=f"ev_ch_mode_{i}_{t}"
            )

            m.addConstr(
                ev_dis[i, t] <= ev_dis_ub * (1 - ev_mode[i, t]),
                name=f"ev_dis_mode_{i}_{t}"
            )

    # Departure SOC target
    for i in I:
        m.addConstr(
            soc_ev[i, int(departure_times[i])] >= soc_target_pcts[i],
            name=f"ev_departure_target_{i}"
        )

    # --------------------------------------------------------
    # Objective
    # --------------------------------------------------------

    m.setObjective(
        gp.quicksum(grid[t] * price[t] for t in T)
        + gp.quicksum(
            (ev_ch[i, t] + ev_dis[i, t]) * ev_deg_cost
            for i in I for t in T
        )
        + gp.quicksum(
            (bat_ch[t] + bat_dis[t]) * bat_deg_cost
            for t in T
        ),
        GRB.MINIMIZE
    )

    # --------------------------------------------------------
    # Solve
    # --------------------------------------------------------

    m.optimize()

    if m.status != GRB.OPTIMAL:
        return None

    # --------------------------------------------------------
    # Extract results
    # --------------------------------------------------------

    grid_profile = np.array([grid[t].X for t in T])
    pv_curt_profile = np.array([pv_curt[t].X for t in T])

    pv_to_load_total = sum(pv_to_load[t].X for t in T)
    pv_to_ev_total = sum(pv_to_ev[t].X for t in T)
    pv_to_bat_total = sum(pv_to_bat[t].X for t in T)

    pv_self_consumption_kWh = pv_to_load_total + pv_to_ev_total + pv_to_bat_total

    if enable_smart_ev:
        ev_charge_total = sum(ev_ch[i, t].X for i in I for t in T)
        ev_discharge_total = sum(ev_dis[i, t].X for i in I for t in T)
        ev_unmet_kWh = 0.0
        ev_unmet_count = 0
    else:
        ev_charge_total = np.sum(unmanaged_ev_charge)
        ev_discharge_total = 0.0
        ev_unmet_kWh = unmanaged_ev_unmet_kWh
        ev_unmet_count = unmanaged_ev_unmet_count

    bat_charge_total = sum(bat_ch[t].X for t in T)
    bat_discharge_total = sum(bat_dis[t].X for t in T)

    battery_cycles = (
        bat_discharge_total / bat_cap_real
        if bat_cap_real > 0
        else 0.0
    )

    result = {
        "total_cost": m.ObjVal,
        "grid_import_total_kWh": np.sum(grid_profile),
        "grid_import_peak_kW": np.max(grid_profile),
        "pv_self_consumption_kWh": pv_self_consumption_kWh,
        "pv_curtailment_kWh": np.sum(pv_curt_profile),

        "ev_charge_total": ev_charge_total,
        "ev_discharge_total": ev_discharge_total,
        "ev_unmet_kWh": ev_unmet_kWh,
        "ev_departure_violation_count": ev_unmet_count,

        "bat_charge_total": bat_charge_total,
        "bat_discharge_total": bat_discharge_total,
        "battery_cycles": battery_cycles
    }

    return result


# ============================================================
# 6. Feature Engineering
# ============================================================

def calculate_ev_peak_available_hours(arrival_times, departure_times, ev_count):
    if ev_count == 0:
        return 0

    count = 0

    for i in range(ev_count):
        for t in PEAK_HOURS:
            if arrival_times[i] <= t < departure_times[i]:
                count += 1

    return count


def build_dataset_row(
    base_case_id,
    scenario_name,
    scenario_config,
    result,
    cost_S0,
    rh,
    bh,
    price,
    weather_type,
    PV_install_rate,
    ev_count,
    bat_count,
    price_gap,
    arrival_times,
    departure_times,
    soc_init_pcts,
    soc_target_pcts
):
    pv_total_kWh = np.sum(rh)
    pv_peak_kW = np.max(rh)
    pv_midday_kWh = np.sum(rh[10:15])

    load_total_kWh = np.sum(bh)
    load_peak_kW = np.max(bh)
    load_peak_time = int(np.argmax(bh))

    bat_cap_real = bat_cap_single * bat_count
    bat_power_kW = bat_dis_single * bat_count

    ev_total_capacity_kWh = ev_count * ev_capacity_single

    avg_arrival_time = np.mean(arrival_times) if ev_count > 0 else 0
    avg_departure_time = np.mean(departure_times) if ev_count > 0 else 0
    avg_parking_hours = np.mean(departure_times - arrival_times) if ev_count > 0 else 0

    avg_initial_soc = np.mean(soc_init_pcts) if ev_count > 0 else 0
    avg_target_soc = np.mean(soc_target_pcts) if ev_count > 0 else 0

    pv_to_load_ratio = pv_total_kWh / load_total_kWh if load_total_kWh > 0 else 0
    battery_to_load_ratio = bat_cap_real / load_total_kWh if load_total_kWh > 0 else 0
    ev_to_load_ratio = ev_total_capacity_kWh / load_total_kWh if load_total_kWh > 0 else 0

    price_gap_ratio = np.max(price) / np.min(price) if np.min(price) > 0 else 0

    parking_flexibility = avg_parking_hours * ev_count
    ev_peak_available_hours = calculate_ev_peak_available_hours(
        arrival_times,
        departure_times,
        ev_count
    )

    pv_surplus_before_storage_kWh = np.sum(np.maximum(0, rh - bh))
    pv_surplus_ratio = (
        pv_surplus_before_storage_kWh / pv_total_kWh
        if pv_total_kWh > 0
        else 0
    )

    storage_to_pv_ratio = bat_cap_real / pv_total_kWh if pv_total_kWh > 0 else 0

    scenario_cost = result["total_cost"]
    incremental_saving = cost_S0 - scenario_cost
    incremental_saving_rate = incremental_saving / cost_S0 if cost_S0 > 0 else 0

    pv_self_consumption_rate = (
        result["pv_self_consumption_kWh"] / pv_total_kWh
        if pv_total_kWh > 0
        else 0
    )

    row = {
        "base_case_id": base_case_id,
        "scenario": scenario_name,

        # Scenario switches
        "battery_enabled": scenario_config["enable_battery"],
        "smart_ev_charging_enabled": scenario_config["enable_smart_ev_charging"],
        "ev_v2b_enabled": scenario_config["enable_ev_discharge"],

        # Main controllable variables
        "weather_type": weather_type,
        "PV_install_rate": PV_install_rate,
        "ev_count": ev_count,
        "bat_count": bat_count,
        "price_gap": price_gap,

        # PV features
        "pv_total_kWh": pv_total_kWh,
        "pv_peak_kW": pv_peak_kW,
        "pv_midday_kWh": pv_midday_kWh,
        "pv_to_load_ratio": pv_to_load_ratio,
        "pv_surplus_before_storage_kWh": pv_surplus_before_storage_kWh,
        "pv_surplus_ratio": pv_surplus_ratio,

        # Load features
        "load_total_kWh": load_total_kWh,
        "load_peak_kW": load_peak_kW,
        "load_peak_time": load_peak_time,

        # Battery features
        "battery_total_capacity_kWh": bat_cap_real,
        "battery_power_kW": bat_power_kW,
        "battery_to_load_ratio": battery_to_load_ratio,
        "storage_to_pv_ratio": storage_to_pv_ratio,

        # EV features
        "ev_total_capacity_kWh": ev_total_capacity_kWh,
        "avg_arrival_time": avg_arrival_time,
        "avg_departure_time": avg_departure_time,
        "avg_parking_hours": avg_parking_hours,
        "avg_initial_soc": avg_initial_soc,
        "avg_target_soc": avg_target_soc,
        "ev_to_load_ratio": ev_to_load_ratio,
        "parking_flexibility": parking_flexibility,
        "ev_peak_available_hours": ev_peak_available_hours,

        # Price features
        "min_price": np.min(price),
        "max_price": np.max(price),
        "actual_price_spread": np.max(price) - np.min(price),
        "price_gap_ratio": price_gap_ratio,

        # Main target
        "pv_only_baseline_cost": cost_S0,
        "scenario_cost": scenario_cost,
        "incremental_saving": incremental_saving,
        "incremental_saving_rate": incremental_saving_rate,

        # Per-unit average value
        "incremental_saving_per_EV": (
            incremental_saving / ev_count if ev_count > 0 else 0
        ),
        "incremental_saving_per_battery": (
            incremental_saving / bat_count if bat_count > 0 else 0
        ),

        # Dispatch outputs
        "grid_import_total_kWh": result["grid_import_total_kWh"],
        "grid_import_peak_kW": result["grid_import_peak_kW"],
        "pv_self_consumption_kWh": result["pv_self_consumption_kWh"],
        "pv_self_consumption_rate": pv_self_consumption_rate,
        "pv_curtailment_kWh": result["pv_curtailment_kWh"],

        # Battery outputs
        "battery_charge_kWh": result["bat_charge_total"],
        "battery_discharge_kWh": result["bat_discharge_total"],
        "battery_cycles": result["battery_cycles"],

        # EV outputs
        "ev_charge_kWh": result["ev_charge_total"],
        "ev_discharge_kWh": result["ev_discharge_total"],
        "ev_unmet_kWh": result["ev_unmet_kWh"],
        "ev_departure_violation_count": result["ev_departure_violation_count"]
    }

    return row


def add_case_level_comparisons(df):
    """
    Add case-level comparison columns:

    battery_extra_saving:
        S1 - S0

    smart_ev_extra_saving:
        S2 - S0

    v2b_extra_over_smart_charging:
        S3 - S2

    integrated_extra_saving:
        S4 - S0

    battery_v2b_interaction_saving:
        S4 - S1 - S3
        because S0 incremental saving = 0
    """

    pivot = df.pivot_table(
        index="base_case_id",
        columns="scenario",
        values="incremental_saving",
        aggfunc="first"
    )

    comp = pd.DataFrame(index=pivot.index)

    comp["battery_extra_saving"] = pivot["S1_PV_battery"]
    comp["smart_ev_extra_saving"] = pivot["S2_PV_smart_EV_charging"]
    comp["v2b_total_extra_saving"] = pivot["S3_PV_V2B"]
    comp["v2b_extra_over_smart_charging"] = (
        pivot["S3_PV_V2B"] - pivot["S2_PV_smart_EV_charging"]
    )
    comp["integrated_extra_saving"] = pivot["S4_PV_battery_V2B"]
    comp["battery_v2b_interaction_saving"] = (
        pivot["S4_PV_battery_V2B"]
        - pivot["S1_PV_battery"]
        - pivot["S3_PV_V2B"]
    )
    comp["S4_vs_best_single_saving"] = (
        pivot["S4_PV_battery_V2B"]
        - np.maximum(pivot["S1_PV_battery"], pivot["S3_PV_V2B"])
    )

    comp = comp.reset_index()

    df = df.merge(comp, on="base_case_id", how="left")

    return df


# ============================================================
# 7. Dataset Generator
# ============================================================

def generate_dataset(
    n_base_cases=1000,
    sunny_weather_file="466920-2026-05-12.csv",
    rainy_weather_file="466920-2026-06-12.csv",
    meter_file="0512電表.csv",
    output_file="v2b_incremental_saving_dataset.csv",
    random_seed=42
):
    np.random.seed(random_seed)

    # --------------------------------------------------------
    # Load fixed data
    # --------------------------------------------------------

    bh = load_building_load(meter_file)

    rh_sunny_base = calculate_pv_from_weather(sunny_weather_file)
    rh_rainy_base = calculate_pv_from_weather(rainy_weather_file)

    print("===== Weather PV Profile Check =====")
    print(f"Sunny PV total: {np.sum(rh_sunny_base):.2f} kWh")
    print(f"Sunny PV peak : {np.max(rh_sunny_base):.2f} kW")
    print(f"Rainy PV total: {np.sum(rh_rainy_base):.2f} kWh")
    print(f"Rainy PV peak : {np.max(rh_rainy_base):.2f} kW")
    print("====================================\n")

    print("===== Building Load Check =====")
    print(f"Load total: {np.sum(bh):.2f} kWh")
    print(f"Load peak : {np.max(bh):.2f} kW")
    print("===============================\n")

    all_rows = []
    valid_base_cases = 0

    # --------------------------------------------------------
    # Monte Carlo base-case loop
    # --------------------------------------------------------

    for s in range(n_base_cases):

        # Controllable base-case variables
        weather_type = np.random.choice([0, 1])
        PV_install_rate = np.random.uniform(0, 1)
        ev_count = np.random.randint(0, 11)
        bat_count = np.random.randint(0, 11)
        price_gap = np.random.uniform(0, 5)

        # Generate data
        price = generate_price(price_gap)

        rh = generate_pv_profile(
            rh_sunny_base=rh_sunny_base,
            rh_rainy_base=rh_rainy_base,
            weather_type=weather_type,
            PV_install_rate=PV_install_rate
        )

        arrival_times, departure_times, soc_init_pcts, soc_target_pcts = generate_ev_data(
            ev_count
        )

        # ----------------------------------------------------
        # S0 baseline: existing PV + unmanaged EV
        # ----------------------------------------------------

        s0_result = calculate_pv_only_unmanaged_baseline(
            rh=rh,
            bh=bh,
            price=price,
            ev_count=ev_count,
            arrival_times=arrival_times,
            departure_times=departure_times,
            soc_init_pcts=soc_init_pcts,
            soc_target_pcts=soc_target_pcts
        )

        cost_S0 = s0_result["total_cost"]

        scenario_results = {
            "S0_PV_only_unmanaged_EV": s0_result
        }

        scenario_configs = {
            "S0_PV_only_unmanaged_EV": {
                "enable_battery": False,
                "enable_smart_ev_charging": False,
                "enable_ev_discharge": False
            }
        }

        # ----------------------------------------------------
        # S1-S4 optimization
        # ----------------------------------------------------

        case_failed = False

        for scenario_name, scenario_config in SCENARIOS.items():

            opt_result = run_v2b_optimization(
                rh=rh,
                bh=bh,
                price=price,
                ev_count=ev_count,
                bat_count=bat_count,
                arrival_times=arrival_times,
                departure_times=departure_times,
                soc_init_pcts=soc_init_pcts,
                soc_target_pcts=soc_target_pcts,
                scenario_config=scenario_config
            )

            if opt_result is None:
                print(f"Base case {s}: {scenario_name} infeasible or no optimal solution")
                case_failed = True
                break

            scenario_results[scenario_name] = opt_result
            scenario_configs[scenario_name] = scenario_config

        if case_failed:
            continue

        # ----------------------------------------------------
        # Save S0-S4 rows
        # ----------------------------------------------------

        for scenario_name, result in scenario_results.items():
            row = build_dataset_row(
                base_case_id=s,
                scenario_name=scenario_name,
                scenario_config=scenario_configs[scenario_name],
                result=result,
                cost_S0=cost_S0,
                rh=rh,
                bh=bh,
                price=price,
                weather_type=weather_type,
                PV_install_rate=PV_install_rate,
                ev_count=ev_count,
                bat_count=bat_count,
                price_gap=price_gap,
                arrival_times=arrival_times,
                departure_times=departure_times,
                soc_init_pcts=soc_init_pcts,
                soc_target_pcts=soc_target_pcts
            )

            all_rows.append(row)

        valid_base_cases += 1

        if (s + 1) % 50 == 0:
            print(f"Finished {s + 1}/{n_base_cases} base cases")

    # --------------------------------------------------------
    # Export dataset
    # --------------------------------------------------------

    df_ml = pd.DataFrame(all_rows)

    if len(df_ml) > 0:
        df_ml = add_case_level_comparisons(df_ml)

    df_ml.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n===== Dataset Generation Finished =====")
    print(f"Valid base cases: {valid_base_cases}")
    print(f"Total rows: {len(df_ml)}")
    print(f"Saved to: {output_file}")

    return df_ml


# ============================================================
# 8. Run
# ============================================================

df_ml = generate_dataset(
    n_base_cases=1000,
    sunny_weather_file="466920-2026-05-12.csv",
    rainy_weather_file="466920-2026-06-12.csv",
    meter_file="0512電表.csv",
    output_file="v2b_incremental_saving_dataset.csv",
    random_seed=42
)

print("\n===== Dataset Head =====")
print(df_ml.head())

print("\n===== Dataset Description =====")
print(df_ml.describe())

===== Weather PV Profile Check =====
Sunny PV total: 1267.75 kWh
Sunny PV peak : 182.52 kW
Rainy PV total: 302.77 kWh
Rainy PV peak : 49.12 kW

===== Building Load Check =====
Load total: 2894.36 kWh
Load peak : 157.90 kW

Finished 50/1000 base cases
Base case 77: S2_PV_smart_EV_charging infeasible or no optimal solution
Finished 100/1000 base cases
Base case 145: S2_PV_smart_EV_charging infeasible or no optimal solution
Finished 150/1000 base cases
Finished 200/1000 base cases
Base case 238: S2_PV_smart_EV_charging infeasible or no optimal solution
Finished 250/1000 base cases
Base case 261: S2_PV_smart_EV_charging infeasible or no optimal solution
Base case 267: S2_PV_smart_EV_charging infeasible or no optimal solution
Base case 270: S2_PV_smart_EV_charging infeasible or no optimal solution
Finished 300/1000 base cases
Base case 335: S2_PV_smart_EV_charging infeasible or no optimal solution
Finished 350/1000 base cases
Base case 356: S2_PV_smart_EV_charging infeasible or no optimal s